In [2]:
from pathlib import Path
from collections import Counter
import hashlib
import os
import json
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm


E:\CondaEnvs\smartagrivision\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Image Extensions
IMAGE_EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".gif",
    ".webp"
)

print(IMAGE_EXTENSIONS)

('.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp')


# Dataset Paths & Input Verification

In [4]:
BASE_DATASET_PATH = Path(
    r"E:\programming languages\DJANGO PROJECT\SmartAgriVision"
    r"\SmartAgriVision\SmartAgriVision_Dataset"
)

FRUITS360_PATH = BASE_DATASET_PATH / "Fruits-360"
NEW_PLANT_PATH = BASE_DATASET_PATH / "New_Plant_Diseases"
PLANTDOC_PATH = BASE_DATASET_PATH / "PlantDoc"
VEGETABLE_PATH = BASE_DATASET_PATH / "Vegetable_Dataset"

datasets = {
    "Fruits-360": FRUITS360_PATH,
    "New Plant Diseases": NEW_PLANT_PATH,
    "PlantDoc": PLANTDOC_PATH,
    "Vegetable Dataset": VEGETABLE_PATH
}


print("DATASET PATH VERIFICATION")


for dataset_name, dataset_path in datasets.items():

    print(f"\nDataset : {dataset_name}")
    print(f"Path    : {dataset_path}")
    print(f"Exists  : {dataset_path.exists()}")
    print(f"Folder  : {dataset_path.is_dir()}")

DATASET PATH VERIFICATION

Dataset : Fruits-360
Path    : E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\SmartAgriVision_Dataset\Fruits-360
Exists  : True
Folder  : True

Dataset : New Plant Diseases
Path    : E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\SmartAgriVision_Dataset\New_Plant_Diseases
Exists  : True
Folder  : True

Dataset : PlantDoc
Path    : E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\SmartAgriVision_Dataset\PlantDoc
Exists  : True
Folder  : True

Dataset : Vegetable Dataset
Path    : E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\SmartAgriVision_Dataset\Vegetable_Dataset
Exists  : True
Folder  : True


In [5]:
# Verification Summary
path_status = []

for dataset_name, dataset_path in datasets.items():

    path_status.append({
        "Dataset": dataset_name,
        "Path": str(dataset_path),
        "Exists": dataset_path.exists(),
        "Is_Directory": dataset_path.is_dir()
    })

df_path_status = pd.DataFrame(path_status)

display(df_path_status)

if df_path_status["Exists"].all():
    print("All dataset paths are valid.")
else:
    print("WARNING: One or more dataset paths are missing.")

,Dataset,Path,Exists,Is_Directory
0,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,True,True
1,New Plant Diseases,E:\programming languages\DJANGO PROJECT\SmartA...,True,True
2,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,True,True
3,Vegetable Dataset,E:\programming languages\DJANGO PROJECT\SmartA...,True,True


All dataset paths are valid.


# Image Inventory

In [6]:
# COLLECT ALL IMAGE PATHS

all_image_records = []

for dataset_name, dataset_path in datasets.items():

    print(f"\nCollecting : {dataset_name}")

    if not dataset_path.exists():
        print("WARNING : Dataset path not found.")
        continue

    for image_path in dataset_path.rglob("*"):

        if not image_path.is_file():
            continue

        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        all_image_records.append({
            "Dataset": dataset_name,
            "Image_Path": str(image_path),
            "Filename": image_path.name,
            "Extension": image_path.suffix.lower()
        })

print()

print(f"Total Images Found : {len(all_image_records):,}")






Total Images Found : 382,601


In [7]:
# Create Inventory DataFrame
image_inventory_df = pd.DataFrame(all_image_records)
print("IMAGE INVENTORY DATAFRAME")
print(f"Total Records : {len(image_inventory_df):,}")
display(image_inventory_df.head(10))

IMAGE INVENTORY DATAFRAME
Total Records : 382,601


,Dataset,Image_Path,Filename,Extension
0,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,r0_103_100.jpg,.jpg
1,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,r0_107_100.jpg,.jpg
2,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,r0_111_100.jpg,.jpg
3,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,r0_115_100.jpg,.jpg
4,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,r0_119_100.jpg,.jpg
5,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,r0_11_100.jpg,.jpg
6,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,r0_123_100.jpg,.jpg
7,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,r0_127_100.jpg,.jpg
8,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,r0_131_100.jpg,.jpg
9,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,r0_135_100.jpg,.jpg


In [8]:
# Dataset-wise Image Count

dataset_image_count = (
    image_inventory_df
    .groupby("Dataset")
    .size()
    .reset_index(name="Images")
)

display(dataset_image_count)
print(
    f"TOTAL IMAGE RECORDS : "
    f"{len(image_inventory_df):,}"
)


,Dataset,Images
0,Fruits-360,182945
1,New Plant Diseases,175734
2,PlantDoc,2922
3,Vegetable Dataset,21000


TOTAL IMAGE RECORDS : 382,601


# Corrupted / Unreadable Image Detection

In [9]:
# SCAN IMAGES
corrupted_images = []
print("SCANNING FOR CORRUPTED / UNREADABLE IMAGES")
for row in tqdm(
    image_inventory_df.itertuples(index=False),
    total=len(image_inventory_df),
    desc="Checking Images",
    unit="image"
):
    image_path = Path(row.Image_Path)

    try:
        with Image.open(image_path) as img:
            img.verify()

    except Exception as e:
        corrupted_images.append({
            "Dataset": row.Dataset,
            "Image_Path": str(image_path),
            "Filename": row.Filename,
            "Error": str(e)
        })

print()
print("CORRUPTED IMAGE SCAN COMPLETED")
print(f"Corrupted / Unreadable Images : {len(corrupted_images):,}")

SCANNING FOR CORRUPTED / UNREADABLE IMAGES


Checking Images: 100%|██████████| 382601/382601 [1:23:24<00:00, 76.46image/s] 


CORRUPTED IMAGE SCAN COMPLETED
Corrupted / Unreadable Images : 0


In [10]:
# Corrupted DataFrame
corrupted_df = pd.DataFrame(corrupted_images)
print("CORRUPTED IMAGE REPORT")

if len(corrupted_df) > 0:
    display(corrupted_df.head(20))
else:
    print("No corrupted or unreadable images found.")

print(f"Total Corrupted Images : {len(corrupted_df):,}")

CORRUPTED IMAGE REPORT
No corrupted or unreadable images found.
Total Corrupted Images : 0


In [11]:
# DATASET-WISE CORRUPTED IMAGE COUNT
if len(corrupted_df) > 0:

    corrupted_summary = (
        corrupted_df
        .groupby("Dataset")
        .size()
        .reset_index(name="Corrupted_Images")
    )

    display(corrupted_summary)

else:

    corrupted_summary = pd.DataFrame(
        columns=["Dataset", "Corrupted_Images"]
    )

    print("No corrupted images detected.")

No corrupted images detected.


# Duplicate Image Detection

In [12]:
# IMAGE HASH FUNCTION
def calculate_image_hash(image_path, chunk_size=1024 * 1024):
    """
    Calculate SHA-256 hash of an image file.
    """

    sha256 = hashlib.sha256()

    with open(image_path, "rb") as file:

        while True:

            chunk = file.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()


In [13]:
# Calculate Hashes
image_hash_records = []

print("=" * 90)
print("CALCULATING IMAGE HASHES")
print("=" * 90)

for row in tqdm(
    image_inventory_df.itertuples(index=False),
    total=len(image_inventory_df),
    desc="Hashing Images",
    unit="image"
):

    image_path = Path(row.Image_Path)

    try:

        file_hash = calculate_image_hash(image_path)

        image_hash_records.append({
            "Dataset": row.Dataset,
            "Image_Path": str(image_path),
            "Filename": row.Filename,
            "Hash": file_hash
        })

    except Exception:
        continue

image_hash_df = pd.DataFrame(image_hash_records)

print()
print("IMAGE HASHING COMPLETED")
print(f"Successfully Hashed : {len(image_hash_df):,}")

CALCULATING IMAGE HASHES


Hashing Images: 100%|██████████| 382601/382601 [1:14:18<00:00, 85.82image/s] 



IMAGE HASHING COMPLETED
Successfully Hashed : 382,601


In [14]:
# Find Duplicates
duplicate_groups = (
    image_hash_df
    .groupby("Hash")
    .size()
    .reset_index(name="Count")
)

duplicate_groups = duplicate_groups[
    duplicate_groups["Count"] > 1
].sort_values(
    "Count",
    ascending=False
)

print("=" * 90)
print("DUPLICATE IMAGE ANALYSIS")
print("=" * 90)

print(
    f"Unique Duplicate Groups : "
    f"{len(duplicate_groups):,}"
)

print(
    f"Duplicate Image Records : "
    f"{duplicate_groups['Count'].sub(1).sum():,}"
)

display(duplicate_groups.head(20))

DUPLICATE IMAGE ANALYSIS
Unique Duplicate Groups : 87,854
Duplicate Image Records : 87,906


,Hash,Count
126840,6e1610d4e0f6f84108af112322d11779254b2b2d419584...,4
63540,372bc4a6e18895011eaa18638acf8cdd83db9c5909af02...,4
70119,3cec67e9fe608677780a3549d893894cad1e7e3f557886...,4
67231,3a60a7b6f3a287981d21dca37f27e5480cc46d1144688d...,4
81199,468b12cc10b2411eab77918cd36de0a7d0005b47361125...,4
33655,1d202e8c6b21351c851a4220c6ac5be10c4bf7f659fa82...,4
48354,2a05a74db3a270ac0ec77c00ebe89a332eed5898194351...,4
136515,767bbeee0c319d4097d7a6d949c6d21def97ceb2d16068...,4
147483,7ff8410db407008bb9aecf0cc5505f2b47ffd89e508413...,4
49400,2aeb7479e975df37dfb54aa7b403f57f2ad90b018c3780...,4


In [15]:
# Duplicate Image Details
duplicate_hashes = set(
    duplicate_groups["Hash"]
)

duplicate_images_df = image_hash_df[
    image_hash_df["Hash"].isin(duplicate_hashes)
].sort_values("Hash")

print("=" * 90)
print("DUPLICATE IMAGE DETAILS")
print("=" * 90)

display(duplicate_images_df.head(50))

print(
    f"Total Duplicate Records : "
    f"{len(duplicate_images_df):,}"
)

DUPLICATE IMAGE DETAILS


,Dataset,Image_Path,Filename,Hash
289134,New Plant Diseases,E:\programming languages\DJANGO PROJECT\SmartA...,ae7425b4-aaae-44e0-ae8e-19623a66ed70___GCREC_B...,00022e991956630b8ae95e7575bda7a96e54b72615378f...
289133,New Plant Diseases,E:\programming languages\DJANGO PROJECT\SmartA...,ae7425b4-aaae-44e0-ae8e-19623a66ed70___GCREC_B...,00022e991956630b8ae95e7575bda7a96e54b72615378f...
326345,New Plant Diseases,E:\programming languages\DJANGO PROJECT\SmartA...,ea119d19-1b3a-4767-af1d-e8b148a3bb8d___FREC_C....,0002811ffd5484e96fd125e098b166bdeb006a9d28e440...
326346,New Plant Diseases,E:\programming languages\DJANGO PROJECT\SmartA...,ea119d19-1b3a-4767-af1d-e8b148a3bb8d___FREC_C....,0002811ffd5484e96fd125e098b166bdeb006a9d28e440...
255275,New Plant Diseases,E:\programming languages\DJANGO PROJECT\SmartA...,5d0dec6f-047f-4049-b9fb-0d3842d6a5cb___JR_HL 8...,00047f1a302165b5ebe3e4fb2334217cf899012f60cef7...
255276,New Plant Diseases,E:\programming languages\DJANGO PROJECT\SmartA...,5d0dec6f-047f-4049-b9fb-0d3842d6a5cb___JR_HL 8...,00047f1a302165b5ebe3e4fb2334217cf899012f60cef7...
341975,New Plant Diseases,E:\programming languages\DJANGO PROJECT\SmartA...,d8a79539-88dc-45a9-9b42-90511c8e79e0___JR_HL 8...,000500883984ba2f90f2753e3d1eea3ebad06aa987e517...
341976,New Plant Diseases,E:\programming languages\DJANGO PROJECT\SmartA...,d8a79539-88dc-45a9-9b42-90511c8e79e0___JR_HL 8...,000500883984ba2f90f2753e3d1eea3ebad06aa987e517...
206975,New Plant Diseases,E:\programming languages\DJANGO PROJECT\SmartA...,5bff6ee9-60ba-46d9-82a6-637458ebb8b0___FREC_Pw...,0005177019652cf1fec23bd69997bee59583d6da32281e...
206976,New Plant Diseases,E:\programming languages\DJANGO PROJECT\SmartA...,5bff6ee9-60ba-46d9-82a6-637458ebb8b0___FREC_Pw...,0005177019652cf1fec23bd69997bee59583d6da32281e...


Total Duplicate Records : 175,760


# Invalid Image Detection

In [16]:
# IMAGE PROPERTY VALIDATION
invalid_images = []

print("CHECKING INVALID IMAGE PROPERTIES")
for row in tqdm(
    image_inventory_df.itertuples(index=False),
    total=len(image_inventory_df),
    desc="Validating Images",
    unit="image"
):

    image_path = Path(row.Image_Path)

    try:

        file_size = image_path.stat().st_size

        with Image.open(image_path) as img:

            width, height = img.size
            mode = img.mode

        problems = []

        if width <= 0 or height <= 0:
            problems.append("Invalid dimensions")

        if file_size == 0:
            problems.append("Zero file size")

        if not problems:
            continue

        invalid_images.append({
            "Dataset": row.Dataset,
            "Image_Path": str(image_path),
            "Filename": row.Filename,
            "Width": width,
            "Height": height,
            "Mode": mode,
            "File_Size_Bytes": file_size,
            "Problem": "; ".join(problems)
        })

    except Exception as e:

        invalid_images.append({
            "Dataset": row.Dataset,
            "Image_Path": str(image_path),
            "Filename": row.Filename,
            "Width": None,
            "Height": None,
            "Mode": None,
            "File_Size_Bytes": (
                image_path.stat().st_size
                if image_path.exists()
                else 0
            ),
            "Problem": f"Validation failed: {e}"
        })

print()
print("INVALID IMAGE CHECK COMPLETED")
print(f"Invalid Images : {len(invalid_images):,}")

CHECKING INVALID IMAGE PROPERTIES


Validating Images: 100%|██████████| 382601/382601 [02:06<00:00, 3029.66image/s]


INVALID IMAGE CHECK COMPLETED
Invalid Images : 0


In [17]:
# Invalid Image Report
invalid_images_df = pd.DataFrame(invalid_images)

print("=" * 90)
print("INVALID IMAGE REPORT")
print("=" * 90)

if len(invalid_images_df) > 0:

    display(invalid_images_df.head(50))

else:

    print("No invalid images detected.")

print(
    f"Total Invalid Images : "
    f"{len(invalid_images_df):,}"
)

INVALID IMAGE REPORT
No invalid images detected.
Total Invalid Images : 0


In [18]:
# CLEANING CHECK SUMMARY
print("DATA CLEANING INITIAL CHECK SUMMARY")
print(
    f"Total Images        : "
    f"{len(image_inventory_df):,}"
)

print(
    f"Corrupted Images    : "
    f"{len(corrupted_df):,}"
)

print(
    f"Duplicate Records   : "
    f"{len(duplicate_images_df):,}"
)

print(
    f"Invalid Images      : "
    f"{len(invalid_images_df):,}"
)


DATA CLEANING INITIAL CHECK SUMMARY
Total Images        : 382,601
Corrupted Images    : 0
Duplicate Records   : 175,760
Invalid Images      : 0


# Resolution & Dimension Cleaning

In [19]:
# IMAGE DIMENSION ANALYSIS
dimension_records = []


print("ANALYZING IMAGE DIMENSIONS")


for row in tqdm(
    image_inventory_df.itertuples(index=False),
    total=len(image_inventory_df),
    desc="Reading Dimensions",
    unit="image"
):

    image_path = Path(row.Image_Path)

    try:

        with Image.open(image_path) as img:

            width, height = img.size

        dimension_records.append({
            "Dataset": row.Dataset,
            "Image_Path": str(image_path),
            "Filename": row.Filename,
            "Width": width,
            "Height": height,
            "Resolution": f"{width} x {height}",
            "Aspect_Ratio": round(width / height, 3)
        })

    except Exception:
        continue

resolution_df = pd.DataFrame(dimension_records)

print()
print("DIMENSION ANALYSIS COMPLETED")
print(f"Valid Dimension Records : {len(resolution_df):,}")

ANALYZING IMAGE DIMENSIONS


Reading Dimensions: 100%|██████████| 382601/382601 [01:48<00:00, 3540.45image/s]



DIMENSION ANALYSIS COMPLETED
Valid Dimension Records : 382,601


In [20]:
# Dimension Statistics
print("IMAGE DIMENSION STATISTICS")
display(
    resolution_df[
        ["Width", "Height", "Aspect_Ratio"]
    ].describe()
)

IMAGE DIMENSION STATISTICS


,Width,Height,Aspect_Ratio
count,382601.000000,382601.000000,382601.000000
mean,184.659525,183.808443,1.001360
std,120.881143,110.746973,0.031465
min,100.000000,69.000000,0.339000
25%,100.000000,100.000000,1.000000
50%,224.000000,224.000000,1.000000
75%,256.000000,256.000000,1.000000
max,6000.000000,6000.000000,4.747000


In [21]:
# Small Images
MIN_DIMENSION = 100

small_images_df = resolution_df[
    (resolution_df["Width"] < MIN_DIMENSION) |
    (resolution_df["Height"] < MIN_DIMENSION)
].copy()


print("SMALL IMAGE ANALYSIS")


print(
    f"Images Smaller Than {MIN_DIMENSION}px "
    f"in Any Dimension : {len(small_images_df):,}"
)

display(small_images_df.head(20))

SMALL IMAGE ANALYSIS
Images Smaller Than 100px in Any Dimension : 3


,Dataset,Image_Path,Filename,Width,Height,Resolution,Aspect_Ratio
358702,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Apple Scab Leaf_4.jpg,115,85,115 x 85,1.353
358896,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Tomato leaf mosaic virus_8.jpg,130,69,130 x 69,1.884
361402,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,train_Tomato mold leaf_4.jpg,115,85,115 x 85,1.353


In [22]:
# FIND VERY LARGE IMAGES
MAX_DIMENSION = 1000

large_images_df = resolution_df[
    (resolution_df["Width"] >= MAX_DIMENSION) |
    (resolution_df["Height"] >= MAX_DIMENSION)
].copy()


print("LARGE IMAGE ANALYSIS")


print(
    f"Images With Dimension >= "
    f"{MAX_DIMENSION}px : {len(large_images_df):,}"
)

display(large_images_df.head(20))

LARGE IMAGE ANALYSIS
Images With Dimension >= 1000px : 1,025


,Dataset,Image_Path,Filename,Width,Height,Resolution,Aspect_Ratio
358679,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Apple leaf_1.jpg,2988,3054,2988 x 3054,0.978
358680,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Apple leaf_2.jpg,2768,2768,2768 x 2768,1.000
358681,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Apple leaf_3.jpg,2988,2988,2988 x 2988,1.000
358684,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Apple leaf_6.jpg,1300,1174,1300 x 1174,1.107
358685,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Apple leaf_7.jpg,1300,957,1300 x 957,1.358
358686,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Apple leaf_8.jpg,2160,2160,2160 x 2160,1.000
358688,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Apple rust leaf_1.jpg,3406,2988,3406 x 2988,1.140
358689,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Apple rust leaf_10.jpg,2448,2576,2448 x 2576,0.950
358695,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Apple rust leaf_7.jpg,3264,2448,3264 x 2448,1.333
358713,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Bell_pepper leaf_6.jpg,2147,1432,2147 x 1432,1.499


In [23]:
# FIND NON-SQUARE IMAGES
non_square_df = resolution_df[
    resolution_df["Width"] != resolution_df["Height"]
].copy()


print("NON-SQUARE IMAGE ANALYSIS")


print(
    f"Total Non-Square Images : "
    f"{len(non_square_df):,}"
)

print(
    f"Percentage Non-Square   : "
    f"{len(non_square_df) / len(resolution_df) * 100:.2f}%"
)

display(non_square_df.head(20))

NON-SQUARE IMAGE ANALYSIS
Total Non-Square Images : 2,147
Percentage Non-Square   : 0.56%


,Dataset,Image_Path,Filename,Width,Height,Resolution,Aspect_Ratio
358679,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Apple leaf_1.jpg,2988,3054,2988 x 3054,0.978
358682,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Apple leaf_4.jpg,800,594,800 x 594,1.347
358683,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Apple leaf_5.jpg,800,692,800 x 692,1.156
358684,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Apple leaf_6.jpg,1300,1174,1300 x 1174,1.107
358685,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Apple leaf_7.jpg,1300,957,1300 x 957,1.358
358688,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Apple rust leaf_1.jpg,3406,2988,3406 x 2988,1.140
358689,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Apple rust leaf_10.jpg,2448,2576,2448 x 2576,0.950
358690,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Apple rust leaf_2.jpg,155,206,155 x 206,0.752
358691,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Apple rust leaf_3.jpg,600,400,600 x 400,1.500
358692,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,test_Apple rust leaf_4.jpg,500,375,500 x 375,1.333


# Image Mode Cleaning

In [24]:
#  IMAGE MODE ANALYSIS
mode_records = []


print("ANALYZING IMAGE MODES")


for row in tqdm(
    image_inventory_df.itertuples(index=False),
    total=len(image_inventory_df),
    desc="Checking Modes",
    unit="image"
):

    image_path = Path(row.Image_Path)

    try:

        with Image.open(image_path) as img:

            mode = img.mode

        mode_records.append({
            "Dataset": row.Dataset,
            "Image_Path": str(image_path),
            "Filename": row.Filename,
            "Mode": mode
        })

    except Exception:
        continue

image_mode_df = pd.DataFrame(mode_records)

print()
print("IMAGE MODE ANALYSIS COMPLETED")
print(f"Mode Records : {len(image_mode_df):,}")

ANALYZING IMAGE MODES


Checking Modes: 100%|██████████| 382601/382601 [01:43<00:00, 3680.40image/s]



IMAGE MODE ANALYSIS COMPLETED
Mode Records : 382,601


In [25]:
# Mode Distribution
mode_summary = (
    image_mode_df
    .groupby("Mode")
    .size()
    .reset_index(name="Images")
    .sort_values("Images", ascending=False)
)


print("IMAGE MODE DISTRIBUTION")
display(mode_summary)

IMAGE MODE DISTRIBUTION


,Mode,Images
2,RGB,382590
0,CMYK,5
3,RGBA,5
1,L,1


In [26]:
# Non-RGB Images
non_rgb_df = image_mode_df[
    image_mode_df["Mode"] != "RGB"
].copy()
print("NON-RGB IMAGE ANALYSIS")
print(
    f"Non-RGB Images : "
    f"{len(non_rgb_df):,}"
)

display(non_rgb_df.head(20))

NON-RGB IMAGE ANALYSIS
Non-RGB Images : 11


,Dataset,Image_Path,Filename,Mode
359103,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,train_Apple rust leaf_79.jpg,CMYK
359105,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,train_Apple rust leaf_9.jpg,L
359456,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,train_Corn Gray leaf spot_15.jpg,CMYK
359483,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,train_Corn Gray leaf spot_40.jpg,RGBA
359722,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,train_Corn rust leaf_27.jpg,RGBA
359726,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,train_Corn rust leaf_30.jpg,RGBA
359730,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,train_Corn rust leaf_34.jpg,RGBA
359803,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,train_grape leaf_10.jpg,CMYK
359897,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,train_grape leaf black rot_33.jpg,CMYK
360596,PlantDoc,E:\programming languages\DJANGO PROJECT\SmartA...,train_Squash Powdery mildew leaf_15.jpg,CMYK


# Class / Folder Cleaning

In [27]:
# CLASS FOLDER DETECTION
class_records = []

print("ANALYZING CLASS / FOLDER STRUCTURE")
for row in image_inventory_df.itertuples(index=False):

    image_path = Path(row.Image_Path)

    try:

        relative_path = image_path.relative_to(
            datasets[row.Dataset]
        )

        parts = relative_path.parts

        if len(parts) >= 2:
            class_name = parts[-2]
        else:
            class_name = "UNKNOWN"

        class_records.append({
            "Dataset": row.Dataset,
            "Image_Path": str(image_path),
            "Class": class_name
        })

    except Exception:
        continue

class_df = pd.DataFrame(class_records)

print()
print("CLASS FOLDER ANALYSIS COMPLETED")
print(f"Class Records : {len(class_df):,}")

ANALYZING CLASS / FOLDER STRUCTURE

CLASS FOLDER ANALYSIS COMPLETED
Class Records : 382,601


In [28]:
# CLASS IMAGE COUNT
class_summary = (
    class_df
    .groupby(["Dataset", "Class"])
    .size()
    .reset_index(name="Images")
    .sort_values(
        ["Dataset", "Images"],
        ascending=[True, False]
    )
)

print("CLASS IMAGE SUMMARY")
display(class_summary.head(30))

CLASS IMAGE SUMMARY


,Dataset,Class,Images
110,Fruits-360,Grape Blue 1,1312
96,Fruits-360,Cucumber 6,1302
66,Fruits-360,Cherimoya 1,1292
62,Fruits-360,Carambola 3,1280
101,Fruits-360,Dates 2,1278
260,Fruits-360,orange 4,1264
180,Fruits-360,Pear 9,1236
162,Fruits-360,Peach 3,1233
208,Fruits-360,Plum 3,1204
147,Fruits-360,Onion 2,1020


In [29]:
# VERY SMALL CLASSES
MIN_CLASS_IMAGES = 2

small_classes_df = class_summary[
    class_summary["Images"] < MIN_CLASS_IMAGES
].copy()


print("SMALL CLASS ANALYSIS")


print(
    f"Classes With Fewer Than "
    f"{MIN_CLASS_IMAGES} Images : "
    f"{len(small_classes_df):,}"
)

display(small_classes_df)

SMALL CLASS ANALYSIS
Classes With Fewer Than 2 Images : 0


,Dataset,Class,Images


In [30]:
# CLASS NAME CHECK
class_name_summary = (
    class_df
    .groupby("Class")["Dataset"]
    .nunique()
    .reset_index(name="Dataset_Count")
)

shared_class_names_df = class_name_summary[
    class_name_summary["Dataset_Count"] > 1
].copy()

print("SHARED CLASS NAME ANALYSIS")
print(
    f"Class Names Appearing in Multiple Datasets : "
    f"{len(shared_class_names_df):,}"
)

display(shared_class_names_df.head(30))

SHARED CLASS NAME ANALYSIS
Class Names Appearing in Multiple Datasets : 0


,Class,Dataset_Count


# Train / Test / Validation Cleaning

In [31]:
# SPLIT DETECTION
split_records = []


print("ANALYZING TRAIN / TEST / VALIDATION SPLITS")


for row in image_inventory_df.itertuples(index=False):

    image_path = Path(row.Image_Path)

    try:

        relative_path = image_path.relative_to(
            datasets[row.Dataset]
        )

        parts_lower = [
            part.lower()
            for part in relative_path.parts
        ]

        if "train" in parts_lower:
            split = "Train"

        elif "test" in parts_lower:
            split = "Test"

        elif "valid" in parts_lower or "validation" in parts_lower:
            split = "Validation"

        else:
            split = "Unknown"

        split_records.append({
            "Dataset": row.Dataset,
            "Image_Path": str(image_path),
            "Split": split
        })

    except Exception:
        continue

split_df = pd.DataFrame(split_records)

print()
print("SPLIT ANALYSIS COMPLETED")
print(f"Split Records : {len(split_df):,}")

ANALYZING TRAIN / TEST / VALIDATION SPLITS

SPLIT ANALYSIS COMPLETED
Split Records : 382,601


In [32]:
# SPLIT DISTRIBUTION
split_summary = (
    split_df
    .groupby(["Dataset", "Split"])
    .size()
    .reset_index(name="Images")
)

print("TRAIN / TEST / VALIDATION SUMMARY")


display(split_summary)

TRAIN / TEST / VALIDATION SUMMARY


,Dataset,Split,Images
0,Fruits-360,Test,45724
1,Fruits-360,Unknown,137221
2,New Plant Diseases,Train,140590
3,New Plant Diseases,Validation,35144
4,PlantDoc,Test,252
5,PlantDoc,Train,2670
6,Vegetable Dataset,Test,3000
7,Vegetable Dataset,Train,15000
8,Vegetable Dataset,Validation,3000


In [33]:
#  UNKNOWN SPLIT CHECK
unknown_split_df = split_df[
    split_df["Split"] == "Unknown"
].copy()


print("UNKNOWN SPLIT ANALYSIS")


print(
    f"Images With Unknown Split : "
    f"{len(unknown_split_df):,}"
)

display(unknown_split_df.head(20))

UNKNOWN SPLIT ANALYSIS
Images With Unknown Split : 137,221


,Dataset,Image_Path,Split
45724,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,Unknown
45725,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,Unknown
45726,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,Unknown
45727,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,Unknown
45728,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,Unknown
45729,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,Unknown
45730,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,Unknown
45731,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,Unknown
45732,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,Unknown
45733,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,Unknown


In [34]:
# CLEANING CANDIDATE SUMMARY
print("SECTION 7–10 CLEANING CANDIDATE SUMMARY")
print(
    f"Total Resolution Records       : "
    f"{len(resolution_df):,}"
)

print(
    f"Small Images (< {MIN_DIMENSION}px)    : "
    f"{len(small_images_df):,}"
)

print(
    f"Large Images (>= {MAX_DIMENSION}px)   : "
    f"{len(large_images_df):,}"
)

print(
    f"Non-Square Images               : "
    f"{len(non_square_df):,}"
)

print(
    f"Non-RGB Images                  : "
    f"{len(non_rgb_df):,}"
)

print(
    f"Total Class Records             : "
    f"{len(class_summary):,}"
)

print(
    f"Small Classes                   : "
    f"{len(small_classes_df):,}"
)

print(
    f"Unknown Split Images            : "
    f"{len(unknown_split_df):,}"
)


SECTION 7–10 CLEANING CANDIDATE SUMMARY
Total Resolution Records       : 382,601
Small Images (< 100px)    : 3
Large Images (>= 1000px)   : 1,025
Non-Square Images               : 2,147
Non-RGB Images                  : 11
Total Class Records             : 342
Small Classes                   : 0
Unknown Split Images            : 137,221


# Cleaning Actions

In [35]:
# CLEANING RULES
MIN_WIDTH = 100
MIN_HEIGHT = 100

# False রাখলে non-RGB image delete হবে না
REMOVE_NON_RGB = False

# False রাখলে non-square image delete হবে না
REMOVE_NON_SQUARE = False

print("CLEANING RULES")


print(f"Minimum Width       : {MIN_WIDTH}px")
print(f"Minimum Height      : {MIN_HEIGHT}px")
print(f"Remove Non-RGB      : {REMOVE_NON_RGB}")
print(f"Remove Non-Square   : {REMOVE_NON_SQUARE}")

CLEANING RULES
Minimum Width       : 100px
Minimum Height      : 100px
Remove Non-RGB      : False
Remove Non-Square   : False


In [36]:
# CREATE CLEANING CANDIDATES
cleaning_records = []

corrupted_paths = set(
    corrupted_df["Image_Path"].astype(str)
) if len(corrupted_df) > 0 else set()

invalid_paths = set(
    invalid_images_df["Image_Path"].astype(str)
) if len(invalid_images_df) > 0 else set()

duplicate_paths = set(
    duplicate_images_df["Image_Path"].astype(str)
) if len(duplicate_images_df) > 0 else set()

non_rgb_paths = set(
    non_rgb_df["Image_Path"].astype(str)
) if len(non_rgb_df) > 0 else set()

small_paths = set(
    small_images_df["Image_Path"].astype(str)
) if len(small_images_df) > 0 else set()

non_square_paths = set(
    non_square_df["Image_Path"].astype(str)
) if len(non_square_df) > 0 else set()


for row in resolution_df.itertuples(index=False):

    image_path = str(row.Image_Path)

    reasons = []

    if image_path in corrupted_paths:
        reasons.append("Corrupted")

    if image_path in invalid_paths:
        reasons.append("Invalid")

    if image_path in duplicate_paths:
        reasons.append("Duplicate")

    if image_path in small_paths:
        reasons.append("Small_Dimension")

    if REMOVE_NON_RGB and image_path in non_rgb_paths:
        reasons.append("Non_RGB")

    if REMOVE_NON_SQUARE and image_path in non_square_paths:
        reasons.append("Non_Square")

    cleaning_records.append({
        "Dataset": row.Dataset,
        "Image_Path": image_path,
        "Width": row.Width,
        "Height": row.Height,
        "Resolution": row.Resolution,
        "Aspect_Ratio": row.Aspect_Ratio,
        "Remove": len(reasons) > 0,
        "Reason": "; ".join(reasons) if reasons else "Keep"
    })


cleaning_df = pd.DataFrame(cleaning_records)


print("CLEANING CANDIDATES CREATED")


print(
    f"Total Records : {len(cleaning_df):,}"
)

print(
    f"Keep          : "
    f"{(~cleaning_df['Remove']).sum():,}"
)

print(
    f"Remove        : "
    f"{cleaning_df['Remove'].sum():,}"
)

display(
    cleaning_df[cleaning_df["Remove"]].head(30)
)

CLEANING CANDIDATES CREATED
Total Records : 382,601
Keep          : 206,838
Remove        : 175,763


,Dataset,Image_Path,Width,Height,Resolution,Aspect_Ratio,Remove,Reason
63692,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,100,100,100 x 100,1.0,True,Duplicate
63934,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,100,100,100 x 100,1.0,True,Duplicate
86536,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,100,100,100 x 100,1.0,True,Duplicate
86623,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,100,100,100 x 100,1.0,True,Duplicate
142282,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,100,100,100 x 100,1.0,True,Duplicate
142283,Fruits-360,E:\programming languages\DJANGO PROJECT\SmartA...,100,100,100 x 100,1.0,True,Duplicate
182945,New Plant Diseases,E:\programming languages\DJANGO PROJECT\SmartA...,256,256,256 x 256,1.0,True,Duplicate
182946,New Plant Diseases,E:\programming languages\DJANGO PROJECT\SmartA...,256,256,256 x 256,1.0,True,Duplicate
182947,New Plant Diseases,E:\programming languages\DJANGO PROJECT\SmartA...,256,256,256 x 256,1.0,True,Duplicate
182948,New Plant Diseases,E:\programming languages\DJANGO PROJECT\SmartA...,256,256,256 x 256,1.0,True,Duplicate


In [37]:
# CLEANING ACTION SUMMARY

print("CLEANING ACTION SUMMARY")


print(
    f"Total Images       : "
    f"{len(cleaning_df):,}"
)

print(
    f"Images To Keep     : "
    f"{(~cleaning_df['Remove']).sum():,}"
)

print(
    f"Images To Remove   : "
    f"{cleaning_df['Remove'].sum():,}"
)



if cleaning_df["Remove"].any():

    reason_summary = (
        cleaning_df[
            cleaning_df["Remove"]
        ]["Reason"]
        .str.split("; ")
        .explode()
        .value_counts()
        .reset_index()
    )

    reason_summary.columns = [
        "Reason",
        "Images"
    ]

    display(reason_summary)

CLEANING ACTION SUMMARY
Total Images       : 382,601
Images To Keep     : 206,838
Images To Remove   : 175,763


,Reason,Images
0,Duplicate,175760
1,Small_Dimension,3


# Clean Dataset Generation

In [43]:
import shutil
# Create cleaned dataset root
CLEANED_DATASET_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# Start with all valid image records
kept_df = image_inventory_df.copy()

# Remove corrupted images
if len(corrupted_df) > 0:
    corrupted_paths = set(
        corrupted_df["Image_Path"].astype(str)
    )

    kept_df = kept_df[
        ~kept_df["Image_Path"].astype(str).isin(
            corrupted_paths
        )
    ].copy()


# Remove invalid images
if len(invalid_images_df) > 0:
    invalid_paths = set(
        invalid_images_df["Image_Path"].astype(str)
    )

    kept_df = kept_df[
        ~kept_df["Image_Path"].astype(str).isin(
            invalid_paths
        )
    ].copy()


# Remove duplicate records
# Keep only the first image from each duplicate hash group
if len(image_hash_df) > 0:

    duplicate_keep_paths = set()

    for _, group in image_hash_df.groupby("Hash"):

        first_path = group.iloc[0]["Image_Path"]

        duplicate_keep_paths.add(
            str(first_path)
        )

    duplicate_paths = set(
        duplicate_images_df["Image_Path"].astype(str)
    ) if len(duplicate_images_df) > 0 else set()

    duplicate_remove_paths = (
        duplicate_paths - duplicate_keep_paths
    )

    kept_df = kept_df[
        ~kept_df["Image_Path"].astype(str).isin(
            duplicate_remove_paths
        )
    ].copy()


# Generate cleaned dataset


copy_summary = []

for dataset_name, dataset_path in datasets.items():

    dataset_output = (
        CLEANED_DATASET_ROOT / dataset_name
    )

    dataset_output.mkdir(
        parents=True,
        exist_ok=True
    )

    dataset_paths = (
        kept_df.loc[
            kept_df["Dataset"] == dataset_name,
            "Image_Path"
        ]
        .astype(str)
        .tolist()
    )

    print()
    print(f"Copying : {dataset_name}")
    print(f"Images  : {len(dataset_paths):,}")

    copied = 0
    failed = 0

    for image_path in tqdm(
        dataset_paths,
        desc=dataset_name,
        unit="image"
    ):

        image_path = Path(image_path)

        try:

            # Preserve original folder structure
            relative_path = image_path.relative_to(
                dataset_path
            )

            destination = (
                dataset_output / relative_path
            )

            destination.parent.mkdir(
                parents=True,
                exist_ok=True
            )

            # Skip if same file already exists
            if (
                destination.exists()
                and destination.stat().st_size
                == image_path.stat().st_size
            ):
                copied += 1
                continue

            shutil.copy2(
                image_path,
                destination
            )

            copied += 1

        except Exception:
            failed += 1

    copy_summary.append({
        "Dataset": dataset_name,
        "Source_Images": len(dataset_paths),
        "Copied_or_Existing": copied,
        "Copy_Failed": failed
    })


SECTION 12 : CLEAN DATASET GENERATION

Copying : Fruits-360
Images  : 182,942


Fruits-360: 100%|██████████| 182942/182942 [01:20<00:00, 2268.41image/s]



Copying : New Plant Diseases
Images  : 87,841


New Plant Diseases: 100%|██████████| 87841/87841 [25:12<00:00, 58.10image/s] 



Copying : PlantDoc
Images  : 2,916


PlantDoc: 100%|██████████| 2916/2916 [00:01<00:00, 1848.16image/s]



Copying : Vegetable Dataset
Images  : 20,996


Vegetable Dataset: 100%|██████████| 20996/20996 [00:09<00:00, 2287.25image/s]


In [44]:
copy_summary_df = pd.DataFrame(
    copy_summary
)

print()
print("=" * 90)
print("CLEAN DATASET GENERATION COMPLETED")
print("=" * 90)

display(copy_summary_df)

print(
    f"Cleaned Dataset Path : "
    f"{CLEANED_DATASET_ROOT.resolve()}"
)


CLEAN DATASET GENERATION COMPLETED


,Dataset,Source_Images,Copied_or_Existing,Copy_Failed
0,Fruits-360,182942,182942,0
1,New Plant Diseases,87841,87841,0
2,PlantDoc,2916,2916,0
3,Vegetable Dataset,20996,20996,0


Cleaned Dataset Path : E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\Cleaned_Dataset


# Post-Cleaning Validation

In [45]:
print("SECTION 13 : POST-CLEANING VALIDATION")
post_clean_records = []

for dataset_name, dataset_path in datasets.items():

    cleaned_path = (
        CLEANED_DATASET_ROOT / dataset_name
    )

    count = 0

    if cleaned_path.exists():

        count = sum(
            1
            for p in cleaned_path.rglob("*")
            if (
                p.is_file()
                and p.suffix.lower() in IMAGE_EXTENSIONS
            )
        )

    post_clean_records.append({
        "Dataset": dataset_name,
        "Cleaned_Images": count,
        "Folder_Exists": cleaned_path.exists()
    })

post_clean_df = pd.DataFrame(post_clean_records)

display(post_clean_df)

print()
print(
    f"Datasets Expected : "
    f"{len(datasets)}"
)

print(
    f"Datasets Created  : "
    f"{post_clean_df['Folder_Exists'].sum()}"
)

print(
    f"Total Clean Images : "
    f"{post_clean_df['Cleaned_Images'].sum():,}"
)

missing_cleaned = post_clean_df[
    ~post_clean_df["Folder_Exists"]
]["Dataset"].tolist()

if missing_cleaned:
    raise RuntimeError(
        "Cleaned dataset folder(s) missing: "
        + ", ".join(missing_cleaned)
    )

SECTION 13 : POST-CLEANING VALIDATION


,Dataset,Cleaned_Images,Folder_Exists
0,Fruits-360,182942,True
1,New Plant Diseases,87841,True
2,PlantDoc,2916,True
3,Vegetable Dataset,20996,True



Datasets Expected : 4
Datasets Created  : 4
Total Clean Images : 294,695


# Final Cleaning Report

In [47]:
original_count = len(image_inventory_df)

removed_count = int(
    cleaning_df["Remove"].sum()
)

# Use copy_summary_df generated by Section 12
if "copy_summary_df" in globals():

    cleaned_count = int(
        copy_summary_df["Copied_or_Existing"].sum()
    )

    copy_failed_count = int(
        copy_summary_df["Copy_Failed"].sum()
    )

else:

    cleaned_count = 0
    copy_failed_count = 0


print("FINAL DATA CLEANING REPORT")


print(
    f"Original Images           : "
    f"{original_count:,}"
)

print(
    f"Images Marked For Removal : "
    f"{removed_count:,}"
)

print(
    f"Images Successfully Copied: "
    f"{cleaned_count:,}"
)

print(
    f"Copy Failed               : "
    f"{copy_failed_count:,}"
)

print(
    f"Expected Clean Images     : "
    f"{original_count - removed_count:,}"
)


FINAL DATA CLEANING REPORT
Original Images           : 382,601
Images Marked For Removal : 175,763
Images Successfully Copied: 294,695
Copy Failed               : 0
Expected Clean Images     : 206,838


In [48]:
# CLEANING BREAKDOWN

print("CLEANING BREAKDOWN")


print(
    f"Corrupted Images : "
    f"{len(corrupted_df):,}"
)

print(
    f"Duplicate Records: "
    f"{len(duplicate_images_df):,}"
)

print(
    f"Invalid Images   : "
    f"{len(invalid_images_df):,}"
)

print(
    f"Small Images     : "
    f"{len(small_images_df):,}"
)

print(
    f"Non-RGB Images   : "
    f"{len(non_rgb_df):,}"
)

print(
    f"Non-Square Images: "
    f"{len(non_square_df):,}"
)


CLEANING BREAKDOWN
Corrupted Images : 0
Duplicate Records: 175,760
Invalid Images   : 0
Small Images     : 3
Non-RGB Images   : 11
Non-Square Images: 2,147


In [50]:
# FINAL DATASET STATISTICS
print("FINAL CLEANED DATASET STATISTICS")


final_statistics = []

for dataset_name, dataset_path in datasets.items():

    valid_count = 0
    resolutions = set()
    modes = set()

    widths = []
    heights = []

    if dataset_path.exists():

        image_paths = [
            path
            for path in dataset_path.rglob("*")
            if path.is_file()
            and path.suffix.lower() in IMAGE_EXTENSIONS
        ]

        for image_path in tqdm(
            image_paths,
            desc=f"Analyzing {dataset_name}",
            unit="image"
        ):

            try:

                with Image.open(image_path) as img:

                    width, height = img.size

                    valid_count += 1

                    resolutions.add(
                        f"{width}x{height}"
                    )

                    modes.add(img.mode)

                    widths.append(width)
                    heights.append(height)

            except Exception:
                pass

    if valid_count > 0:

        final_statistics.append({
            "Dataset": dataset_name,
            "Valid Images": valid_count,
            "Unique Resolutions": len(resolutions),
            "Image Modes": ", ".join(
                sorted(modes)
            ),
            "Minimum Width": min(widths),
            "Maximum Width": max(widths),
            "Minimum Height": min(heights),
            "Maximum Height": max(heights)
        })

    else:

        final_statistics.append({
            "Dataset": dataset_name,
            "Valid Images": 0,
            "Unique Resolutions": 0,
            "Image Modes": "None",
            "Minimum Width": None,
            "Maximum Width": None,
            "Minimum Height": None,
            "Maximum Height": None
        })


final_statistics_df = pd.DataFrame(
    final_statistics
)

print()
print("FINAL CLEANED DATASET STATISTICS")
display(final_statistics_df)

FINAL CLEANED DATASET STATISTICS


Analyzing Vegetable Dataset: 100%|██████████| 21000/21000 [04:37<00:00, 75.54image/s]


FINAL CLEANED DATASET STATISTICS


,Dataset,Valid Images,Unique Resolutions,Image Modes,Minimum Width,Maximum Width,Minimum Height,Maximum Height
0,Fruits-360,182945,1,RGB,100,100,100,100
1,New Plant Diseases,175734,1,RGB,256,256,256,256
2,PlantDoc,2922,1763,"CMYK, L, RGB, RGBA",115,6000,69,6000
3,Vegetable Dataset,21000,10,RGB,224,224,187,224


In [52]:
# Get total copied and failed counts 
if "copy_summary_df" in globals():

    successfully_copied = int(
        copy_summary_df["Copied_or_Existing"].sum()
    )

    copy_failed_count = int(
        copy_summary_df["Copy_Failed"].sum()
    )

else:

    successfully_copied = 0
    copy_failed_count = 0


# Final cleaning report

final_cleaning_report = pd.DataFrame({
    "Metric": [
        "Original Images",
        "Corrupted Images",
        "Duplicate Records",
        "Invalid Images",
        "Small Images",
        "Images Marked For Removal",
        "Images Successfully Copied",
        "Copy Failed"
    ],

    "Count": [
        original_count,
        len(corrupted_df),
        len(duplicate_images_df),
        len(invalid_images_df),
        len(small_images_df),
        int(removed_count),
        successfully_copied,
        copy_failed_count
    ]
})


print("FINAL CLEANING REPORT TABLE")

display(final_cleaning_report)

FINAL CLEANING REPORT TABLE


,Metric,Count
0,Original Images,382601
1,Corrupted Images,0
2,Duplicate Records,175760
3,Invalid Images,0
4,Small Images,3
5,Images Marked For Removal,175763
6,Images Successfully Copied,294695
7,Copy Failed,0


In [53]:
# FINAL STATUS

print("DATA CLEANING COMPLETED")

# Get copy failure count from Section 12
if "copy_summary_df" in globals():

    total_copy_failed = int(
        copy_summary_df["Copy_Failed"].sum()
    )

else:

    total_copy_failed = 0


if total_copy_failed == 0:

    print("STATUS : SUCCESS")

else:

    print("STATUS : REVIEW REQUIRED")


print()

print(
    "Cleaned Dataset Location:"
)

print(
    CLEANED_DATASET_ROOT.resolve()
)

DATA CLEANING COMPLETED
STATUS : SUCCESS

Cleaned Dataset Location:
E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\Cleaned_Dataset
